# Malaria Cell Classification with a Residual Network and Grad-CAM

Binary classification of thin blood smear cell images (**parasitized** vs **uninfected**) on the full NIH Malaria Cell Images dataset, followed by Grad-CAM to see which image regions drive each prediction.

**Credit.** The network architecture and training setup follow Adrian Rosebrock's PyImageSearch tutorial *Deep learning and medical image analysis with Keras* (2018). This notebook ports that code to TensorFlow 2, trains and evaluates on the full dataset with a fixed train/validation/test split, and adds Grad-CAM interpretability.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [ ]:
import os, json, random
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none found - switch the runtime to a GPU")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Configuration

In [ ]:
IMG_SIZE = 64        # the tutorial used 64x64; try 96 or 128 later to see if detail helps
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 0.1  # SGD with momentum, cosine-decayed over training

## 3. Load the full dataset

TensorFlow Datasets hosts the NIH dataset (about 27,500 images, roughly balanced), so there is nothing to download by hand. It has a single split, which is divided 80/10/10 into training, validation and test sets. The split is deterministic, so every run uses the same test images.

In [ ]:
(train_raw, val_raw, test_raw), info = tfds.load(
    "malaria",
    split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    as_supervised=True,
    with_info=True,
)
CLASS_NAMES = info.features["label"].names
NUM_CLASSES = len(CLASS_NAMES)
n_train, n_val, n_test = (int(d.cardinality()) for d in (train_raw, val_raw, test_raw))
print("Classes:", CLASS_NAMES)
print(f"Train {n_train} | Validation {n_val} | Test {n_test} | Total {n_train + n_val + n_test}")

In [ ]:
# A few raw examples. Note the images come in different sizes.
plt.figure(figsize=(10, 4))
for i, (img, label) in enumerate(train_raw.take(10)):
    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(f"{CLASS_NAMES[int(label)]}\n{img.shape[0]}x{img.shape[1]}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Input pipeline

Images are resized to `IMG_SIZE`, scaled to [0, 1], and the training set gets the same kinds of augmentation as the tutorial (rotation, zoom, shift, horizontal flip). Validation and test images are not augmented.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.one_hot(label, NUM_CLASSES)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(20 / 360),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomFlip("horizontal"),
], name="augmentation")

train_ds = (train_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
            .cache()
            .shuffle(10000, seed=SEED)
            .batch(BATCH_SIZE)
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 5. Model

The same bottleneck ResNet as the tutorial (stages (2, 2, 3), filters (32, 64, 128, 256)), ported from standalone Keras to `tf.keras`. Two small changes:

- **Global average pooling** instead of a fixed 8x8 pool, so the model works at any `IMG_SIZE`.
- **Categorical cross-entropy**, which is the correct loss for a two-class softmax output.

The last activation layer is named `final_relu` so that Grad-CAM can find it.

In [ ]:
from tensorflow.keras import layers, regularizers, Model

def residual_module(x, filters, stride, red=False, reg=1e-4, bn_eps=2e-5, bn_mom=0.9):
    shortcut = x
    # 1x1 -> 3x3 -> 1x1 bottleneck, with pre-activation batch norm
    bn1 = layers.BatchNormalization(epsilon=bn_eps, momentum=bn_mom)(x)
    act1 = layers.Activation("relu")(bn1)
    c1 = layers.Conv2D(int(filters * 0.25), 1, use_bias=False, kernel_regularizer=regularizers.l2(reg))(act1)

    bn2 = layers.BatchNormalization(epsilon=bn_eps, momentum=bn_mom)(c1)
    act2 = layers.Activation("relu")(bn2)
    c2 = layers.Conv2D(int(filters * 0.25), 3, strides=stride, padding="same", use_bias=False,
                       kernel_regularizer=regularizers.l2(reg))(act2)

    bn3 = layers.BatchNormalization(epsilon=bn_eps, momentum=bn_mom)(c2)
    act3 = layers.Activation("relu")(bn3)
    c3 = layers.Conv2D(filters, 1, use_bias=False, kernel_regularizer=regularizers.l2(reg))(act3)

    # project the shortcut when the spatial size or channel count changes
    if red:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False,
                                 kernel_regularizer=regularizers.l2(reg))(act1)
    return layers.add([c3, shortcut])

def build_resnet(size, classes, stages=(2, 2, 3), filters=(32, 64, 128, 256), reg=5e-4,
                 bn_eps=2e-5, bn_mom=0.9):
    inputs = layers.Input(shape=(size, size, 3))
    x = layers.BatchNormalization(epsilon=bn_eps, momentum=bn_mom)(inputs)
    x = layers.Conv2D(filters[0], 5, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(reg))(x)
    x = layers.BatchNormalization(epsilon=bn_eps, momentum=bn_mom)(x)
    x = layers.Activation("relu")(x)
    x = layers.ZeroPadding2D(1)(x)
    x = layers.MaxPooling2D(3, strides=2)(x)

    for i, n_modules in enumerate(stages):
        stride = 1 if i == 0 else 2
        x = residual_module(x, filters[i + 1], stride, red=True, reg=reg, bn_eps=bn_eps, bn_mom=bn_mom)
        for _ in range(n_modules - 1):
            x = residual_module(x, filters[i + 1], 1, reg=reg, bn_eps=bn_eps, bn_mom=bn_mom)

    x = layers.BatchNormalization(epsilon=bn_eps, momentum=bn_mom)(x)
    x = layers.Activation("relu", name="final_relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(classes, activation="softmax", kernel_regularizer=regularizers.l2(reg))(x)
    return Model(inputs, outputs, name="malaria_resnet")

model = build_resnet(IMG_SIZE, NUM_CLASSES)
print(f"Parameters: {model.count_params():,}")

## 6. Train

In [ ]:
steps = EPOCHS * int(np.ceil(n_train / BATCH_SIZE))
schedule = tf.keras.optimizers.schedules.CosineDecay(LEARNING_RATE, decay_steps=steps)
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=schedule, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    # keep the weights from the epoch with the best validation accuracy
    tf.keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/best_model.keras", monitor="val_accuracy",
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.CSVLogger(f"{OUT_DIR}/training_log.csv"),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

In [ ]:
h = history.history
epochs = np.arange(1, len(h["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, h["loss"], label="train"); axes[0].plot(epochs, h["val_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Epoch"); axes[0].legend()
axes[1].plot(epochs, h["accuracy"], label="train"); axes[1].plot(epochs, h["val_accuracy"], label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=150)
plt.show()

## 7. Evaluate on the held-out test set

The test set is used only here, once, with the best checkpoint chosen on validation accuracy. **These are the numbers to report.**

In [ ]:
best = tf.keras.models.load_model(f"{OUT_DIR}/best_model.keras")

y_true, y_prob = [], []
for x, y in test_ds:
    y_prob.append(best.predict(x, verbose=0))
    y_true.append(np.argmax(y.numpy(), axis=1))
y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)
y_pred = np.argmax(y_prob, axis=1)

acc = float(np.mean(y_pred == y_true))
pos = CLASS_NAMES.index("parasitized")
auc = float(roc_auc_score(y_true == pos, y_prob[:, pos]))
print(f"Test accuracy: {acc:.4f}   ROC AUC (parasitized): {auc:.4f}   n = {len(y_true)}\n")
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print(report)

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", values_format="d")
plt.title("Test set confusion matrix")
plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

with open(f"{OUT_DIR}/test_metrics.json", "w") as f:
    json.dump({"test_accuracy": acc, "roc_auc_parasitized": auc, "n_test": int(len(y_true)),
               "confusion_matrix": cm.tolist(), "img_size": IMG_SIZE, "epochs": EPOCHS,
               "split": "tfds malaria train[:80%] / [80%:90%] / [90%:]"}, f, indent=2)
with open(f"{OUT_DIR}/classification_report.txt", "w") as f:
    f.write(report)

## 8. Grad-CAM

Grad-CAM (Selvaraju et al., 2017) weights the last convolutional feature maps by the gradient of the predicted class score, giving a coarse map of the regions the model relied on. For a good malaria classifier, the heat should sit on the parasite (the stained dots inside the cell), not on the cell edge or the background.

In [ ]:
grad_model = Model(best.inputs, [best.get_layer("final_relu").output, best.output])

def grad_cam(image, class_idx=None):
    # image: (H, W, 3) float in [0, 1]. Returns a heatmap in [0, 1] at image size, plus the class used.
    img = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        fmaps, preds = grad_model(img, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, fmaps)
    weights = tf.reduce_mean(grads, axis=(1, 2))                     # one weight per channel
    cam = tf.nn.relu(tf.reduce_sum(fmaps[0] * weights[0], axis=-1))  # weighted sum of feature maps
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], image.shape[:2])[..., 0]
    return cam.numpy(), class_idx

In [ ]:
# Collect test images by outcome: correct parasitized, correct uninfected, and misclassified.
test_images = np.concatenate([x.numpy() for x, _ in test_ds])
unin = CLASS_NAMES.index("uninfected")
groups = {
    "Correct: parasitized": np.where((y_true == pos) & (y_pred == pos))[0][:6],
    "Correct: uninfected": np.where((y_true == unin) & (y_pred == unin))[0][:6],
    "Misclassified": np.where(y_true != y_pred)[0][:6],
}
groups = {k: v for k, v in groups.items() if len(v) > 0}  # skip empty groups

fig, axes = plt.subplots(len(groups) * 2, 6, figsize=(13, 4.6 * len(groups)))
for g, (title, idxs) in enumerate(groups.items()):
    for j in range(6):
        ax_img, ax_cam = axes[2 * g, j], axes[2 * g + 1, j]
        ax_img.axis("off"); ax_cam.axis("off")
        if j >= len(idxs):
            continue
        i = idxs[j]
        cam, c = grad_cam(test_images[i])
        ax_img.imshow(test_images[i])
        ax_img.set_title(f"true {CLASS_NAMES[y_true[i]]}\npred {CLASS_NAMES[c]} ({y_prob[i, c]:.2f})", fontsize=8)
        ax_cam.imshow(test_images[i]); ax_cam.imshow(cam, cmap="jet", alpha=0.45)
    axes[2 * g, 0].text(-0.3, 0.5, title, transform=axes[2 * g, 0].transAxes, rotation=90,
                        va="center", ha="right", fontsize=11, weight="bold")
plt.suptitle("Grad-CAM on test images (top: input, bottom: heatmap)", y=1.0)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/gradcam.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Download the results

In [ ]:
import shutil
shutil.make_archive("malaria_results", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("malaria_results.zip")
except ImportError:
    print("Not running in Colab; results are in", os.path.abspath(OUT_DIR))

The zip contains `test_metrics.json`, `classification_report.txt`, `confusion_matrix.png`, `training_curves.png`, `gradcam.png`, `training_log.csv` and `best_model.keras`.